# INSIDE-OUT: Emotion Recognition through Handwriting Analysis
## CNN-HMM Hybrid Model - Google Colab Pipeline

This notebook connects to your private Google Drive dataset and runs the full pipeline:
1. Mount Google Drive & load dataset
2. Preprocess handwriting images
3. Train CNN feature extractor
4. Train HMM classifier
5. Evaluate with cross-validation

---
## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q hmmlearn scikit-image tqdm

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from hmmlearn import hmm

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 2. Configure Paths

**IMPORTANT:** Upload your dataset to Google Drive with this structure:
```
My Drive/
  EMHA/
    DATA/
      LABELED/
        HAPPY/    <- happy handwriting images
        SAD/      <- sad handwriting images
```

In [ ]:
# === CONFIGURE YOUR PATHS HERE ===
DRIVE_BASE = '/content/drive/My Drive/EMHA/DATA'

LABELED_DIR = os.path.join(DRIVE_BASE, 'LABELED')
HAPPY_DIR = os.path.join(LABELED_DIR, 'HAPPY')
SAD_DIR = os.path.join(LABELED_DIR, 'SAD')

# Local working directories (on Colab's fast storage)
WORK_DIR = '/content/EMHA'
PROCESSED_DIR = os.path.join(WORK_DIR, 'PROCESSED')
MODEL_DIR = os.path.join(WORK_DIR, 'models')

os.makedirs(os.path.join(PROCESSED_DIR, 'HAPPY'), exist_ok=True)
os.makedirs(os.path.join(PROCESSED_DIR, 'SAD'), exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify Drive paths exist
for path, name in [(HAPPY_DIR, 'HAPPY'), (SAD_DIR, 'SAD')]:
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        print(f'{name}: {count} images found')
    else:
        print(f'WARNING: {name} directory not found at {path}')
        print(f'Please create this folder in your Google Drive and add images.')

---
## 3. Preprocessing Pipeline

In [ ]:
class PreprocessingPipeline:
    """Preprocessing pipeline for handwriting images."""

    def __init__(self, target_size=(224, 224), denoise_kernel_size=3):
        self.target_size = target_size
        self.denoise_kernel_size = denoise_kernel_size

    def grayscale(self, image):
        if len(image.shape) == 3:
            return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return image

    def binarize(self, image):
        _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    def denoise(self, image):
        kernel = np.ones((self.denoise_kernel_size, self.denoise_kernel_size), np.uint8)
        opened = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
        return closed

    def correct_skew(self, image):
        coords = np.column_stack(np.where(image > 0))
        if len(coords) < 5:
            return image
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) > 15:
            return image  # Skip extreme angles
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h),
                                 flags=cv2.INTER_CUBIC,
                                 borderMode=cv2.BORDER_REPLICATE)
        return rotated

    def normalize(self, image):
        resized = cv2.resize(image, self.target_size)
        normalized = resized.astype(np.float32) / 255.0
        return normalized

    def process(self, image):
        gray = self.grayscale(image)
        binary = self.binarize(gray)
        denoised = self.denoise(binary)
        deskewed = self.correct_skew(denoised)
        normalized = self.normalize(deskewed)
        return normalized


pipeline = PreprocessingPipeline(target_size=(224, 224))
print('Preprocessing pipeline ready')

In [ ]:
# Preprocess all images from Drive -> local processed directory
def preprocess_dataset(src_dir, dst_dir, label_name):
    """Preprocess images from source to destination."""
    src_path = Path(src_dir)
    dst_path = Path(dst_dir) / label_name
    dst_path.mkdir(parents=True, exist_ok=True)

    extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    image_files = [f for f in src_path.iterdir() if f.suffix.lower() in extensions]

    processed = 0
    failed = 0

    for img_file in tqdm(image_files, desc=f'Processing {label_name}'):
        try:
            image = cv2.imread(str(img_file))
            if image is None:
                failed += 1
                continue
            result = pipeline.process(image)
            out_file = dst_path / f'{img_file.stem}.png'
            cv2.imwrite(str(out_file), (result * 255).astype(np.uint8))
            processed += 1
        except Exception as e:
            print(f'Error processing {img_file.name}: {e}')
            failed += 1

    print(f'{label_name}: {processed} processed, {failed} failed')
    return processed

# Run preprocessing
n_happy = preprocess_dataset(HAPPY_DIR, PROCESSED_DIR, 'HAPPY')
n_sad = preprocess_dataset(SAD_DIR, PROCESSED_DIR, 'SAD')
print(f'\nTotal: {n_happy + n_sad} images preprocessed')

In [ ]:
# Visualize preprocessing results
def show_samples(directory, label, n=5):
    """Show sample preprocessed images."""
    path = Path(directory) / label
    images = sorted(path.glob('*.png'))[:n]

    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
    if len(images) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, images):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap='gray')
        ax.set_title(img_path.stem, fontsize=8)
        ax.axis('off')
    plt.suptitle(f'{label} Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(PROCESSED_DIR, 'HAPPY')
show_samples(PROCESSED_DIR, 'SAD')

---
## 4. Dataset & DataLoader

In [ ]:
class HandwritingDataset(Dataset):
    """PyTorch Dataset for handwriting emotion classification."""

    EMOTION_TO_LABEL = {'HAPPY': 0, 'SAD': 1}
    LABEL_TO_EMOTION = {0: 'HAPPY', 1: 'SAD'}

    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.samples = []
        self._load_samples()

    def _load_samples(self):
        for emotion, label in self.EMOTION_TO_LABEL.items():
            emotion_dir = self.data_dir / emotion
            if emotion_dir.exists():
                for ext in ('*.png', '*.jpg', '*.jpeg'):
                    for img_path in emotion_dir.glob(ext):
                        self.samples.append((str(img_path), label))
        print(f'Loaded {len(self.samples)} samples '
              f'(HAPPY: {sum(1 for _, l in self.samples if l == 0)}, '
              f'SAD: {sum(1 for _, l in self.samples if l == 1)})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('L')  # Grayscale
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_labels(self):
        return [label for _, label in self.samples]


# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load full dataset (we'll split later in cross-validation)
full_dataset = HandwritingDataset(PROCESSED_DIR, transform=val_transform)

# Show class distribution
labels = full_dataset.get_labels()
plt.figure(figsize=(6, 4))
plt.bar(['HAPPY', 'SAD'], [labels.count(0), labels.count(1)], color=['#2ecc71', '#3498db'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.show()

---
## 5. CNN Feature Extractor

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """CNN for extracting features from handwriting images."""

    def __init__(self, input_channels=1, num_features=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, num_features),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x


class EmotionCNN(nn.Module):
    """Full CNN with classifier head (for pretraining the feature extractor)."""

    def __init__(self, input_channels=1, num_features=256, num_classes=2):
        super().__init__()
        self.extractor = CNNFeatureExtractor(input_channels, num_features)
        self.classifier = nn.Linear(num_features, num_classes)

    def forward(self, x):
        features = self.extractor(x)
        logits = self.classifier(features)
        return logits

    def extract_features(self, x):
        self.eval()
        with torch.no_grad():
            return self.extractor(x)


# Quick test
model = EmotionCNN().to(device)
dummy = torch.randn(2, 1, 224, 224).to(device)
out = model(dummy)
print(f'Model output shape: {out.shape}')  # Should be (2, 2)
feats = model.extract_features(dummy)
print(f'Feature shape: {feats.shape}')  # Should be (2, 256)

---
## 6. HMM Classifier

In [ ]:
class HMMClassifier:
    """HMM classifier using separate models per emotion."""

    def __init__(self, n_states=4, n_iter=100, covariance_type='diag'):
        self.n_states = n_states
        self.n_iter = n_iter
        self.covariance_type = covariance_type
        self.models = {}

    def fit(self, features, labels):
        """Train separate HMMs for HAPPY (0) and SAD (1)."""
        for label, name in [(0, 'HAPPY'), (1, 'SAD')]:
            mask = labels == label
            class_features = features[mask]
            n_components = min(self.n_states, len(class_features))

            model = hmm.GaussianHMM(
                n_components=n_components,
                covariance_type=self.covariance_type,
                n_iter=self.n_iter,
                random_state=42
            )
            model.fit(class_features)
            self.models[name] = model
            print(f'  HMM trained for {name}: {mask.sum()} samples, {n_components} states')

    def predict(self, features):
        """Predict emotion for each sample."""
        predictions = []
        confidences = []

        for i in range(len(features)):
            sample = features[i:i+1]
            scores = {}
            for name, model in self.models.items():
                scores[name] = model.score(sample)

            predicted = max(scores, key=scores.get)
            pred_label = 0 if predicted == 'HAPPY' else 1
            predictions.append(pred_label)

            # Confidence via softmax of scores
            vals = np.array(list(scores.values()))
            probs = np.exp(vals - np.max(vals))
            probs = probs / probs.sum()
            confidences.append(float(np.max(probs)))

        return np.array(predictions), np.array(confidences)


print('HMM Classifier ready')

---
## 7. Training Functions

In [ ]:
def train_cnn(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    """Train the CNN model with early stopping."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        train_loss /= total
        train_acc = correct / total

        # Validate
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        val_loss /= total
        val_acc = correct / total

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{epochs} - '
                  f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    # Restore best model
    if best_state:
        model.load_state_dict(best_state)

    return history


def extract_all_features(model, data_loader):
    """Extract CNN features for all samples."""
    model.eval()
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            features = model.extract_features(images)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)


print('Training functions ready')

---
## 8. 5-Fold Cross-Validation (Hybrid CNN-HMM)

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 10
N_FOLDS = 5
HMM_STATES = 4
CNN_FEATURES = 256

# Get all sample paths and labels
all_paths = [s[0] for s in full_dataset.samples]
all_labels = np.array(full_dataset.get_labels())

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []
all_y_true = []
all_y_pred = []

print(f'Starting {N_FOLDS}-Fold Cross-Validation')
print(f'Total samples: {len(all_labels)} (HAPPY: {(all_labels==0).sum()}, SAD: {(all_labels==1).sum()})')
print('=' * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

    # Create subset datasets
    train_subset = torch.utils.data.Subset(full_dataset, train_idx)
    val_subset = torch.utils.data.Subset(full_dataset, val_idx)

    # Apply train augmentation to training set
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

    # Step 1: Train CNN
    print('\n  Step 1: Training CNN...')
    cnn_model = EmotionCNN(input_channels=1, num_features=CNN_FEATURES).to(device)
    history = train_cnn(cnn_model, train_loader, val_loader,
                        epochs=EPOCHS, lr=LEARNING_RATE, patience=PATIENCE)

    # Step 2: Extract features with trained CNN
    print('\n  Step 2: Extracting CNN features...')
    train_features, train_labels = extract_all_features(cnn_model, train_loader)
    val_features, val_labels = extract_all_features(cnn_model, val_loader)
    print(f'  Train features: {train_features.shape}, Val features: {val_features.shape}')

    # Step 3: Train HMM on CNN features
    print('\n  Step 3: Training HMM...')
    hmm_clf = HMMClassifier(n_states=HMM_STATES)
    hmm_clf.fit(train_features, train_labels)

    # Step 4: Evaluate
    y_pred, confidences = hmm_clf.predict(val_features)

    acc = accuracy_score(val_labels, y_pred)
    prec = precision_score(val_labels, y_pred, average='macro', zero_division=0)
    rec = recall_score(val_labels, y_pred, average='macro', zero_division=0)
    f1 = f1_score(val_labels, y_pred, average='macro', zero_division=0)

    fold_results.append({'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1})
    all_y_true.extend(val_labels)
    all_y_pred.extend(y_pred)

    print(f'\n  Fold {fold+1} Results: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}')

print('\n' + '=' * 60)
print('CROSS-VALIDATION SUMMARY')
print('=' * 60)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    values = [r[metric] for r in fold_results]
    print(f'{metric.capitalize():>10}: {np.mean(values):.4f} (+/- {np.std(values):.4f})')

---
## 9. Evaluation & Visualization

In [ ]:
# Classification Report
print('\nFull Classification Report (all folds combined):')
print(classification_report(all_y_true, all_y_pred, target_names=['HAPPY', 'SAD']))

# Confusion Matrix
cm = confusion_matrix(all_y_true, all_y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['HAPPY', 'SAD'],
            yticklabels=['HAPPY', 'SAD'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (All Folds)')
plt.tight_layout()
plt.show()

# Per-fold accuracy bar chart
plt.figure(figsize=(8, 4))
fold_accs = [r['accuracy'] for r in fold_results]
bars = plt.bar(range(1, N_FOLDS + 1), fold_accs, color='#3498db', alpha=0.8)
plt.axhline(y=np.mean(fold_accs), color='red', linestyle='--', label=f'Mean: {np.mean(fold_accs):.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Accuracy per Fold')
plt.ylim(0, 1)
plt.legend()
plt.xticks(range(1, N_FOLDS + 1))
plt.tight_layout()
plt.show()

---
## 10. Save Final Model to Google Drive

In [ ]:
# Train final model on ALL data
print('Training final model on all data...')
final_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

final_cnn = EmotionCNN(input_channels=1, num_features=CNN_FEATURES).to(device)

# Train CNN
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_cnn.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

final_cnn.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0, 0, 0
    for images, labels in final_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/total:.4f}, Acc: {correct/total:.4f}')

# Extract features and train final HMM
all_features, all_labels_arr = extract_all_features(final_cnn, final_loader)
final_hmm = HMMClassifier(n_states=HMM_STATES)
final_hmm.fit(all_features, all_labels_arr)

# Save to Google Drive
save_dir = '/content/drive/My Drive/EMHA/models'
os.makedirs(save_dir, exist_ok=True)

torch.save(final_cnn.state_dict(), os.path.join(save_dir, 'cnn_model.pth'))

import joblib
joblib.dump(final_hmm.models, os.path.join(save_dir, 'hmm_models.pkl'))

# Save cross-validation results
import json
results = {
    'fold_results': fold_results,
    'mean_accuracy': float(np.mean([r['accuracy'] for r in fold_results])),
    'mean_f1': float(np.mean([r['f1'] for r in fold_results])),
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'hmm_states': HMM_STATES,
        'cnn_features': CNN_FEATURES,
        'n_folds': N_FOLDS
    }
}
with open(os.path.join(save_dir, 'cv_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nModels saved to Google Drive: {save_dir}')
print('Files: cnn_model.pth, hmm_models.pkl, cv_results.json')

---
## 11. Predict on New Handwriting Sample

In [ ]:
def predict_emotion(image_path, cnn_model, hmm_classifier):
    """Predict emotion from a handwriting image."""
    # Load and preprocess
    image = cv2.imread(image_path)
    processed = pipeline.process(image)

    # Convert to tensor
    tensor = torch.tensor(processed, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    # Extract features
    cnn_model.eval()
    with torch.no_grad():
        features = cnn_model.extract_features(tensor).cpu().numpy()

    # Classify
    prediction, confidence = hmm_classifier.predict(features)
    emotion = 'HAPPY' if prediction[0] == 0 else 'SAD'

    # Display
    plt.figure(figsize=(6, 4))
    plt.imshow(processed, cmap='gray')
    plt.title(f'Predicted: {emotion} (confidence: {confidence[0]:.2%})', fontsize=14)
    plt.axis('off')
    plt.show()

    return emotion, confidence[0]

# Example usage (uncomment and set path to your image):
# emotion, conf = predict_emotion('/content/drive/My Drive/EMHA/DATA/RAW/test_sample.png', final_cnn, final_hmm)